# Spotify Data Extraction Pipeline

This notebook is the first step of the Spotify Playlist Analyzer project. Its purpose is to collect raw playlist data from the Spotify Web API, convert the API responses into a structured pandas DataFrame, enrich the data with artist metadata, create early language/genre features, and save the final dataset for later EDA, statistical inference, and ML modelling.

The output of this notebook is:

```text
data/df.csv
```

That CSV is used by the later notebooks in the project.

## 1. Import Packages

In [1]:
import os
from pathlib import Path

import requests
import pandas as pd
import time
import re # regular expression library

## 2. Spotify Credentials and Access Token

This section authenticates with Spotify using the Client Credentials flow. The access token is then used to call Spotify API endpoints.

For a public/project version, credentials should not be hard-coded in the notebook. They should be moved to environment variables or a local `.env` file that is not committed to version control.

In [2]:
# Spotify access information loaded from .env

def find_data_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        notebook_data = candidate / "notebooks" / "data"
        local_data = candidate / "data"
        if notebook_data.exists():
            return notebook_data
        if candidate.name == "notebooks" and local_data.exists():
            return local_data
    return Path("data")


DATA_DIR = find_data_dir()
DATA_PATH = DATA_DIR / "df.csv"

def find_env_file():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        env_path = candidate / ".env"
        if env_path.exists():
            return env_path
    return Path(".env")


def load_env_file(path=None):
    env_path = Path(path) if path is not None else find_env_file()
    if not env_path.exists():
        return

    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

load_env_file()

CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID") or os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET") or os.getenv("CLIENT_SECRET")

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Set CLIENT_ID and CLIENT_SECRET in .env before calling Spotify.")

# Obtained using the CLIENT_ID and CLIENT_SECRET
token_url = "https://accounts.spotify.com/api/token"

response = requests.post(
    token_url,
    headers={
        "Content-Type": "application/x-www-form-urlencoded"
    },
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET
    }
)
response.raise_for_status()
token_data = response.json()

ACCESS_TOKEN = token_data["access_token"]

# playlist for testing
playlist_id = '4k4kTCxwnV9sgE0whd3Gg2'

# base url to start with
base_url = 'https://api.spotify.com/v1/playlists/'

# headers to call Spotify API in the required format
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}


## 3. Get Playlist Items

This section calls the Spotify playlist items endpoint and handles pagination. Spotify returns playlist tracks in pages, so the function keeps requesting the `next` URL until all tracks have been collected.

The result is still nested JSON, so the next step converts it into a tabular format.

In [3]:
def get_all_playlist_items(playlist_id):
    """
    Get the corresponding Spotify playlist as JSON data
    
    Parameters:
        playlist_id: string
            a playlist id to query data
    
    Returns:
        dict
            JSON response from Spotify playlist items endpoint.
    """
    url = f"{base_url}{playlist_id}/items"
    all_items = []

    while url is not None:
        response = requests.get(url, headers=headers)
        
        if response.status_code != 200:
            print(response.status_code)
            print(response.text)
            return None
        
        data = response.json()
        all_items.extend(data["items"]) # 把 iterable 里的元素拆开后逐个加入

    # next page url; if this is None, no more pages
        url = data["next"]

    return {"items": all_items}

In [4]:
data = get_all_playlist_items(playlist_id)

### 3.1 Convert Playlist JSON to a DataFrame

The Spotify response contains deeply nested information about tracks, artists, albums, and playlist metadata. This function extracts the most useful fields and stores them in one row per track.

Important fields created here include track ID, track name, artist names, main artist ID, album information, release date, duration, popularity, explicit status, track URL, and the date the song was added to the playlist.

In [5]:
def playlist_items_to_df(data):
    """
    Convert Spotify playlist items JSON data into a clean pandas DataFrame.
    
    Parameters:
        data: dict
            JSON response from Spotify playlist items endpoint.
    
    Returns:
        pd.DataFrame
            A cleaned DataFrame containing track-level information.
    """
    rows = []

    for item in data['items']:
        track = item['track']
        
        # skip if the track is empty or unavailable
        if track is None:
            continue

        artist_names = [artist["name"] for artist in track["artists"]]
        artist_ids = [artist["id"] for artist in track["artists"]]

        rows.append({
            "track_id": track["id"],
            "track_name": track["name"],

            # all artists
            "artist_names": ", ".join(artist_names),
            "artist_ids": ", ".join(artist_ids),

            # main artist
            "main_artist_name": artist_names[0] if artist_names else None,
            "main_artist_id": artist_ids[0] if artist_ids else None,

            # album info
            "album_id": track["album"]["id"],
            "album_name": track["album"]["name"],
            "release_date": track["album"]["release_date"],

            # track features
            "duration_ms": track["duration_ms"],
            "duration_min": track["duration_ms"] / 60000,
            "popularity": track["popularity"],
            "explicit": track["explicit"],

            # urls
            "track_url": track["external_urls"]["spotify"],

            # playlist-level metadata
            "added_at": item["added_at"],
            "is_local": item["is_local"]
        })

    return pd.DataFrame(rows)


In [6]:
df = playlist_items_to_df(data)
df.shape

(238, 16)

In [7]:
df.head()

,track_id,track_name,artist_names,artist_ids,main_artist_name,main_artist_id,album_id,album_name,release_date,duration_ms,duration_min,popularity,explicit,track_url,added_at,is_local
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,5lpy0FxmneKqFb4zeoiSHM,十八般武藝,2010-08-12,251146,4.185767,50,False,https://open.spotify.com/track/0VGPId3riSeBV5F...,2025-11-20T07:03:00Z,False
1,6aF2RVTPJSJCM1MpUjne5X,落葉歸根,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,1lxlQW182pklrfpD93faq1,改變自己,2007-07-01,315226,5.253767,51,False,https://open.spotify.com/track/6aF2RVTPJSJCM1M...,2025-11-20T07:03:00Z,False
2,68RtalgSqaCbeZN42QeMS0,花田錯,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,34D46J9tIGCqAj3FeiEA9O,蓋世英雄,2005-12-30,227960,3.799333,50,False,https://open.spotify.com/track/68RtalgSqaCbeZN...,2025-11-20T07:03:00Z,False
3,3agtg0x11wPvLIWkYR39nZ,Somewhere I Belong,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,4Gfnly5CzMJQqkUFfoHaP3,Meteora,2003-03-25,213933,3.565550,81,False,https://open.spotify.com/track/3agtg0x11wPvLIW...,2025-11-20T07:03:00Z,False
4,57BrRMwf9LrcmuOsyGilwr,Crawling,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,6hPkbAV3ZXpGZBGUvL6jVM,Hybrid Theory (Bonus Edition),2000-10-24,208960,3.482667,84,False,https://open.spotify.com/track/57BrRMwf9LrcmuO...,2025-11-20T07:03:00Z,False


## 4. Artist Metadata and Genre Features

The playlist endpoint gives track and album information, but genre information is stored at the artist level in Spotify's API. This section queries each unique main artist and adds artist metadata such as:

- artist genres
- artist popularity
- artist follower count

These features will later be useful for EDA, statistical analysis, and recommendation models.

In [8]:
def get_artist_info(artist_id, headers):
    """
    Get artist metadata from Spotify Artist API.
    """
    url = f"https://api.spotify.com/v1/artists/{artist_id}"
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed for artist_id={artist_id}")
        print(response.status_code)
        print(response.text)
        return None
    
    return response.json()

In [9]:
def get_artist_metadata_df(df, headers, sleep_time=0.1):
    """
    Get Spotify artist metadata for all unique main artists in the playlist DataFrame.
    
    Returns:
        artist_df: DataFrame with artist-level metadata.
    """
    unique_artist_ids = df["main_artist_id"].dropna().unique()
    
    rows = []
    
    for i, artist_id in enumerate(unique_artist_ids):
        artist = get_artist_info(artist_id, headers)
        
        if artist is None:
            continue
        
        rows.append({
            "main_artist_id": artist["id"],
            "artist_name_from_api": artist["name"],
            "main_artist_genres": artist["genres"],
            "artist_popularity": artist["popularity"],
            "artist_followers": artist["followers"]["total"]
        })
        
        # Optional: show progress every 20 artists
        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1}/{len(unique_artist_ids)} artists")
        
        time.sleep(sleep_time)
    
    artist_df = pd.DataFrame(rows)
    return artist_df

In [10]:
artist_df = get_artist_metadata_df(df, headers)

Processed 20/134 artists
Processed 40/134 artists
Processed 60/134 artists
Processed 80/134 artists
Processed 100/134 artists
Processed 120/134 artists


In [11]:
artist_df

,main_artist_id,artist_name_from_api,main_artist_genres,artist_popularity,artist_followers
0,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,"[mandopop, c-pop, taiwanese pop, chinese r&b, ...",57,1192346
1,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,"[nu metal, rap metal, rock, alternative metal]",89,34542635
2,40tNK2YedBV2jRFAHxpifB,David Tao,"[mandopop, chinese r&b, c-pop, taiwanese pop, ...",60,1270568
3,4Kxlr1PRlDKEB0ekOCyHgX,BIGBANG,[k-pop],71,6530949
4,1r9DuPTHiQ7hnRRZ99B8nL,JOLIN,"[mandopop, c-pop, taiwanese pop, malay pop]",61,1229596
...,...,...,...,...,...
129,6noxsCszBEEK04kCehugOp,A-Mei Chang,"[mandopop, taiwanese pop, c-pop, malay pop]",61,1293497
130,24jrxG0tKcwgAzsLuPzyMi,Namewee,"[mandopop, c-pop, taiwanese pop, chinese hip h...",46,394992
131,7I6BcL7hpJB7DATY9uPXcH,四季音色,[],26,3640
132,21z8to3YxZXgKYJpBB54P2,Daniel Seavey,[],56,245272


In [12]:
# merge the song_df and artist_df
df_enriched = df.merge(
    artist_df,
    on="main_artist_id",
    how="left"
)

df_enriched.shape

(238, 20)

## 5. Language and Script Features

Spotify does not directly provide the sung language of each track in this dataset. As an early approximation, this section detects the writing system used in the track title, such as Chinese, Japanese, Korean, Latin, mixed, or unknown.

This is not the same as true song language. For example, a Latin-script title may not always mean the song is in English. However, title script is still a useful exploratory feature for this playlist because it captures part of the multilingual structure of the data.

In [13]:
import re

def detect_title_script(title):
    """
    Detect the writing system used in a song title.

    Returns:
        chinese / japanese / korean / latin / mixed / unknown
    """
    if not isinstance(title, str) or title.strip() == "":
        return "unknown"
    
    has_chinese = bool(re.search(r"[\u4e00-\u9fff]", title))
    has_japanese = bool(re.search(r"[\u3040-\u30ff]", title))
    has_korean = bool(re.search(r"[\uac00-\ud7af]", title))
    has_latin = bool(re.search(r"[A-Za-z]", title))
    
    scripts = []
    
    if has_chinese:
        scripts.append("chinese")
    if has_japanese:
        scripts.append("japanese")
    if has_korean:
        scripts.append("korean")
    if has_latin:
        scripts.append("latin")
    
    if len(scripts) == 0:
        return "unknown"
    elif len(scripts) == 1:
        return scripts[0]
    else:
        return "mixed"

In [14]:
def map_title_script_to_label(script):
    """
    Convert title script into a readable feature label.
    """
    mapping = {
        "chinese": "Chinese-script title",
        "japanese": "Japanese-script title",
        "korean": "Korean-script title",
        "latin": "Latin-script title",
        "mixed": "Mixed-script title",
        "unknown": "Unknown-script title"
    }
    
    return mapping.get(script, "Unknown-script title")

In [15]:
df_enriched["title_script"] = df_enriched["track_name"].apply(detect_title_script)

In [16]:
df_enriched["song_title_language"] = df_enriched["title_script"].apply(
    map_title_script_to_label
)

In [17]:
df_enriched.head()

,track_id,track_name,artist_names,artist_ids,main_artist_name,main_artist_id,album_id,album_name,release_date,duration_ms,...,explicit,track_url,added_at,is_local,artist_name_from_api,main_artist_genres,artist_popularity,artist_followers,title_script,song_title_language
0,0VGPId3riSeBV5FTRn850F,需要人陪,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,5lpy0FxmneKqFb4zeoiSHM,十八般武藝,2010-08-12,251146,...,False,https://open.spotify.com/track/0VGPId3riSeBV5F...,2025-11-20T07:03:00Z,False,Leehom Wang,"[mandopop, c-pop, taiwanese pop, chinese r&b, ...",57,1192346,chinese,Chinese-script title
1,6aF2RVTPJSJCM1MpUjne5X,落葉歸根,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,1lxlQW182pklrfpD93faq1,改變自己,2007-07-01,315226,...,False,https://open.spotify.com/track/6aF2RVTPJSJCM1M...,2025-11-20T07:03:00Z,False,Leehom Wang,"[mandopop, c-pop, taiwanese pop, chinese r&b, ...",57,1192346,chinese,Chinese-script title
2,68RtalgSqaCbeZN42QeMS0,花田錯,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,Leehom Wang,2F5W6Rsxwzg0plQ0w8dSyt,34D46J9tIGCqAj3FeiEA9O,蓋世英雄,2005-12-30,227960,...,False,https://open.spotify.com/track/68RtalgSqaCbeZN...,2025-11-20T07:03:00Z,False,Leehom Wang,"[mandopop, c-pop, taiwanese pop, chinese r&b, ...",57,1192346,chinese,Chinese-script title
3,3agtg0x11wPvLIWkYR39nZ,Somewhere I Belong,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,4Gfnly5CzMJQqkUFfoHaP3,Meteora,2003-03-25,213933,...,False,https://open.spotify.com/track/3agtg0x11wPvLIW...,2025-11-20T07:03:00Z,False,Linkin Park,"[nu metal, rap metal, rock, alternative metal]",89,34542635,latin,Latin-script title
4,57BrRMwf9LrcmuOsyGilwr,Crawling,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,Linkin Park,6XyY86QOPPrYVGvF9ch6wz,6hPkbAV3ZXpGZBGUvL6jVM,Hybrid Theory (Bonus Edition),2000-10-24,208960,...,False,https://open.spotify.com/track/57BrRMwf9LrcmuO...,2025-11-20T07:03:00Z,False,Linkin Park,"[nu metal, rap metal, rock, alternative metal]",89,34542635,latin,Latin-script title


In [18]:
df_enriched["song_title_language"].value_counts()

song_title_language
Latin-script title       156
Chinese-script title      72
Mixed-script title         9
Japanese-script title      1
Name: count, dtype: int64

### 5.1 Combine Genre and Title-Script Signals

This rule-based feature tries to create broader cultural/language-region tags by combining artist genres and title script. Regional genres such as `mandopop`, `k-pop`, or `j-pop` are treated as stronger signals than broad genres such as `pop` or `rock`.

This feature is intentionally heuristic. It is useful for exploration, but it should be interpreted carefully and can be improved later with better language metadata or lyric-language detection.

In [19]:
def get_language_tags_from_genres_and_title(row):
    """
    Estimate language/cultural-region tags using artist genres and title script.
    
    Rule:
    - Regional/language-specific genres have priority.
    - General Western style genres are only used when no regional tag is found.
    """
    genres = row["main_artist_genres"]
    title_script = row["title_script"]
    
    tags = []
    
    chinese_keywords = [
        "mandopop", "c-pop", "cantopop", "taiwanese pop",
        "chinese", "chinese r&b", "chinese hip hop"
    ]
    
    korean_keywords = [
        "k-pop", "korean", "korean pop", "korean r&b"
    ]
    
    japanese_keywords = [
        "j-pop", "japanese", "japanese pop", "anime"
    ]
    
    western_keywords = [
        "rock", "metal", "alternative", "indie", "punk",
        "folk", "soul", "country", "edm", "electronic", "dance",
        "hip hop", "rap", "r&b", "pop"
    ]
    
    if isinstance(genres, list) and len(genres) > 0:
        genre_text = " ".join(genres).lower()
        
        # First: check strong regional/language signals
        if any(word in genre_text for word in chinese_keywords):
            tags.append("Chinese-associated")
        
        if any(word in genre_text for word in korean_keywords):
            tags.append("Korean-associated")
        
        if any(word in genre_text for word in japanese_keywords):
            tags.append("Japanese-associated")
        
        # Only use Western-associated if no regional/language tag was found
        if len(tags) == 0 and any(word in genre_text for word in western_keywords):
            tags.append("Western-associated")
    
    # Fallback based on title script
    if len(tags) == 0:
        if title_script == "latin":
            tags.append("Western/Latin-title-associated")
        elif title_script == "chinese":
            tags.append("Chinese-associated")
        elif title_script == "korean":
            tags.append("Korean-associated")
        elif title_script == "japanese":
            tags.append("Japanese-associated")
        elif title_script == "mixed":
            tags.append("Mixed-associated")
        else:
            tags.append("Unknown")
    
    return tags

In [20]:
df_enriched["genre_language_tags"] = df_enriched.apply(
    get_language_tags_from_genres_and_title,
    axis=1
)


In [21]:
df_enriched['genre_language_tags'].value_counts()

genre_language_tags
[Chinese-associated]                       85
[Western/Latin-title-associated]           78
[Western-associated]                       51
[Korean-associated]                        11
[Japanese-associated]                      11
[Chinese-associated, Korean-associated]     2
Name: count, dtype: int64

## 6. Save the Enriched DataFrame

The enriched DataFrame is saved as a CSV file so the later notebooks can focus on analysis rather than repeated API calls.

Saving the data also makes the project more reproducible during exploration, because EDA and modelling can run from a stable snapshot instead of calling the API every time.

In [22]:
DATA_PATH.parent.mkdir(exist_ok=True)
df_enriched.to_csv(DATA_PATH, index=False)

## 7. Pipeline Summary and Next Steps

This notebook produces the project dataset by combining three layers of information:

1. **Track-level playlist data** from the Spotify playlist endpoint.
2. **Artist-level metadata** from the Spotify artist endpoint.
3. **Engineered features** based on title script and genre-language heuristics.

The final dataset is ready for:

- EDA and dashboard visualizations
- statistical inference about popularity differences
- regression analysis
- ML modelling and recommendation experiments

Main limitations to remember:

- The title-language feature is based on writing system, not confirmed sung language.
- Artist genres are Spotify-provided metadata and may be broad or incomplete.
- Spotify popularity reflects Spotify's platform ecosystem, not universal music popularity.
- This is a personal playlist sample, so later conclusions should be interpreted as user-specific rather than general population claims.